# Kafka issue types: TF-IDF + neural net

In [ ]:
import re
import pandas as pd
from jira import JIRA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm.auto import tqdm
import numpy as np

jira = JIRA('https://issues.apache.org/jira')

jql = 'project=KAFKA'

all_issues = []
start = 0
batch_size = 100

while True:
    issues = jira.search_issues(
        jql,
        startAt=start,
        maxResults=batch_size,
        fields=['summary', 'issuetype', 'status', 'created', 'reporter', 'description'])
    if not issues:
        break
    all_issues.extend(issues)
    print(f"Found {len(all_issues)} issues")
    start += batch_size
print(f"Found {len(all_issues)} issues total")


def extract_issue(issue):
    return {
        'bug': issue.key,
        'summary': issue.fields.summary or '',
        'description': issue.fields.description or '',
        'type': issue.fields.issuetype.name,
        'status': issue.fields.status.name,
        'created': issue.fields.created,
        'reporter': issue.fields.reporter.displayName if issue.fields.reporter else 'Unknown'
    }


data = [extract_issue(issue) for issue in all_issues]
df = pd.DataFrame(data)
df.to_csv('Kafka_issues.csv', index=False)

print(f"\nTotal issues: {len(df)}")
df['text'] = (df['summary'].fillna('') + ' ' + df['description'].fillna('')).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)

MIN_CLASS_COUNT = 50
type_counts = df['type'].value_counts()
rare_types = type_counts[type_counts < MIN_CLASS_COUNT].index
if len(rare_types) > 0:
    print(f"Collapsing rare issue types into 'Other' (<50 ex.): {list(rare_types)}")
    df['type'] = df['type'].apply(lambda t: 'Other' if t in rare_types else t)

categories = sorted(df['type'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(categories)}
df['label'] = df['type'].map(label_to_idx)

print(f"\nModeling categories: {categories}")
print(df['type'].value_counts())

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    stop_words='english',
    ngram_range=(1, 2),
    lowercase=True,
    token_pattern=r'\b[A-Za-z]+\b',
)

x_train_tfidf = vectorizer.fit_transform(train_texts).toarray()
x_test_tfidf = vectorizer.transform(test_texts).toarray()

print(f"TF-IDF feature shape: {x_train_tfidf.shape}")

class TfidfDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = TfidfDataset(x_train_tfidf, train_labels)
test_dataset = TfidfDataset(x_test_tfidf, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class SimpleNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.3):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.out = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        x = self.dropout(self.relu(self.bn3(self.fc3(x))))
        x = self.out(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SimpleNN(x_train_tfidf.shape[1], len(categories)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
print("Starting training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for features, labels in progress:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        output = model(features)
        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels)
macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=categories))

Found 19454 issues total

Total issues: 19454

Modeling categories: ['Bug', 'Improvement', 'New Feature', 'Sub-task', 'Task', 'Test', 'Wish']
type
Bug            8877
Improvement    5004
Sub-task       3315
Task            871
Test            648
New Feature     606
Wish            133
Name: count, dtype: int64
TF-IDF feature shape: (15563, 5000)
Starting training...


Epoch 1/10, Loss: 1.2146


Epoch 2/10, Loss: 0.9226


Epoch 3/10, Loss: 0.7670


Epoch 4/10, Loss: 0.6321


Epoch 5/10, Loss: 0.5230


Epoch 6/10, Loss: 0.4210


Epoch 7/10, Loss: 0.3593


Epoch 8/10, Loss: 0.3058


Epoch 9/10, Loss: 0.2621


Epoch 10/10, Loss: 0.2356

Accuracy: 0.5988
Macro F1: 0.3629
Weighted F1: 0.5840

Classification Report:
              precision    recall  f1-score   support

         Bug       0.71      0.79      0.75      1775
 Improvement       0.48      0.43      0.46      1001
 New Feature       0.25      0.17      0.20       121
    Sub-task       0.54      0.61      0.57       663
        Task       0.34      0.16      0.21       174
        Test       0.39      0.31      0.34       130
        Wish       0.00      0.00      0.00        27

    accuracy                           0.60      3891
   macro avg       0.39      0.35      0.36      3891
weighted avg       0.58      0.60      0.58      3891

